Import the dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("CADCODER/GenCAD-Code")

train = ds['train']
test = ds['test']
eval = ds['validation']

Download the checkpoint (if continuing training)

In [ ]:
from huggingface_hub import create_repo
import os

HUB_REPO = ""
os.environ["HF_TOKEN"] = ""
checkpoint = "sft"

create_repo(repo_id=HUB_REPO, private=True, exist_ok=True)

if checkpoint == "sft" :
    from gdrive_fsspec import GoogleDriveFileSystem

    fs = GoogleDriveFileSystem(use_listings_cache=False, skip_instance_cache=True, 
                               auth_kwargs={"use_local_webserver": False})
    fs.get("drive_file_path", "local_path", recursive=True)


else:
    from huggingface_hub import hf_hub_download, snapshot_download

    # revision = ""
    # path = snapshot_download(repo_id = HUB_REPO, revision = revision, local_dir = local_dir)
    path = snapshot_download(repo_id= HUB_REPO,allow_patterns=f"{checkpoint}/*", local_dir=f"RL_checkpoint/{checkpoint}")
    print(path)

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
device = 'cuda'

Set target modules (trainable params about 0.6% for Qwen3.5-4B in this case)

In [ ]:
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "in_proj_qkv", "out_proj",
    "gate_proj", "up_proj", "down_proj"
]

import accelerate
from transformers import AutoProcessor, AutoModelForMultimodalLM
from peft import LoraConfig, get_peft_model

processor = AutoProcessor.from_pretrained("Qwen/Qwen3.5-4B")
model = AutoModelForMultimodalLM.from_pretrained("Qwen/Qwen3.5-4B", device_map="auto")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

Rounded off dataset as it had values up to 13-15 decimal places (only kept 4 decimal places), all e values were less than e^-13, kept 0

In [ ]:
import re

compiled_pattern = re.compile(r'(?<![a-zA-Z_])(-?\d+\.\d+(?:[eE][+-]?\d+)?)')

def round_off(match):
    num_str = match.group(0)
    num = float(num_str)
    if 'e' in num_str.lower():
        return '0.0'
    else:
        return str(round(num, 4))

In [ ]:
def apply_rounding(example):
    example["cadquery"] = compiled_pattern.sub(round_off, example["cadquery"])
    return example

In [ ]:
train_rounded = train.map(apply_rounding)
eval_for_training = eval.shuffle(seed=42).select(range(200))
eval_training_rounded = eval_for_training.map(apply_rounding)

To extract only the python code and nothing else from what the model returns:

In [ ]:
_CODE_FENCE_RE = re.compile(r'```(?:python)?\s*\n?(.*?)\n?```', re.DOTALL)

def extract_code(text):
    match = _CODE_FENCE_RE.search(text)
    if match:
        return match.group(1).strip()
    return text.strip()

In [ ]:
_sample_img = next(iter(train_rounded))["image"]
_prompt_msg = [{"role": "user", "content": [
    {"type": "image", "image": _sample_img},
    {"type": "text", "text": "Generate the CadQuery Python code to create this 3D CAD model. Return only the code, no explanation."}
]}]
_prompt_text = processor.apply_chat_template(_prompt_msg, tokenize=False, add_generation_prompt=True, enable_thinking=False)
FIXED_PROMPT_LEN = processor(text=[_prompt_text], images=[_sample_img], return_tensors="pt")["input_ids"].shape[1]
print(FIXED_PROMPT_LEN)

In [ ]:
def collate_fn(examples):
    full_texts = []
    images = []
    for ex in examples:
        user_msg = [{"role": "user", "content": [
            {"type": "image", "image": ex["image"]},
            {"type": "text", "text": "Generate the CadQuery Python code to create this 3D CAD model. Return only the code, no explanation."}
        ]}]
        full_msg = user_msg + [{"role": "assistant", "content": ex["cadquery"]}]
        full_texts.append(processor.apply_chat_template(full_msg, tokenize=False, enable_thinking=False))
        images.append(ex["image"])

    batch = processor(text=full_texts, images=images, padding=True, return_tensors="pt")

    labels = batch["input_ids"].clone()
    attn = batch["attention_mask"]
    for i in range(len(examples)):
        mask_row = attn[i]
        real_positions = mask_row.nonzero(as_tuple=True)[0]
        first_real = real_positions[0].item()
        labels[i, mask_row == 0] = -100
        labels[i, first_real:first_real + FIXED_PROMPT_LEN] = -100

    batch["labels"] = labels
    return batch

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
    learning_rate=2e-4,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    bf16=True,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=3,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    num_train_epochs=3
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_rounded,
    eval_dataset=eval_training_rounded,
    data_collator=collate_fn,

)

trainer.train(resume_from_checkpoint = "")